# Module 3 — Intent Classifier (Few-Shot LLM)
**Mental Health Support Chatbot**

Classifies the user's intent into one of 5 classes using **few-shot prompting**:

| Intent | When |
|---|---|
| `greeting` | "hi", "hello", "good morning" |
| `goodbye` | "bye", "see you", "take care" |
| `gratitude` | "thank you", "thanks a lot" |
| `asking_mental_health_question` | Any real mental health query |
| `out_of_scope` | Anything unrelated to mental health |

## 1. Install

In [ ]:
!pip install -q groq

## 2. Imports

In [1]:
import os, json, re
from groq import Groq


## 3. Groq API key

In [ ]:
GROQ_API_KEY = "groq_api"

client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready.")


Groq client ready.


## 4. Few-shot prompt

In [5]:
SYSTEM_PROMPT = """You are an intent classifier for a mental health support chatbot.
Classify the user message into exactly one of these intents:
- greeting
- goodbye
- gratitude
- asking_mental_health_question
- out_of_scope

Reply with ONLY the intent label, nothing else.

Examples:
User: Hello there! -> greeting
User: Hey, good morning -> greeting
User: Goodbye, take care -> goodbye
User: See you later -> goodbye
User: Thank you so much -> gratitude
User: Thanks, that really helped -> gratitude
User: I have been feeling very anxious lately and cannot sleep -> asking_mental_health_question
User: How do I deal with depression and negative thoughts? -> asking_mental_health_question
User: What is the best recipe for pasta? -> out_of_scope
User: Who won the football match yesterday? -> out_of_scope
"""

def classify_intent(text: str) -> str:
    """Returns one of the 5 intent labels."""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": text},
        ],
        temperature=0,
        max_tokens=10,
    )
    raw = response.choices[0].message.content.strip().lower()

    valid = {"greeting","goodbye","gratitude","asking_mental_health_question","out_of_scope"}
    return raw if raw in valid else "out_of_scope"


## 5. Quick test

In [6]:
test_cases = [
    ("Hi! How are you?",                                    "greeting"),
    ("Goodbye, thank you!",                                 "goodbye"),
    ("Thanks so much for your help",                        "gratitude"),
    ("I feel very anxious and I cannot sleep at night",     "asking_mental_health_question"),
    ("How do I manage panic attacks?",                      "asking_mental_health_question"),
    ("What is the capital of France?",                      "out_of_scope"),
    ("أنا أشعر بالاكتئاب الشديد",                          "asking_mental_health_question"),
]

print(f"{'Input':<45} {'Expected':<35} {'Predicted':<35} {'OK'}")
print("-" * 120)
correct = 0
for text, expected in test_cases:
    predicted = classify_intent(text)
    ok = "✓" if predicted == expected else "✗"
    if predicted == expected:
        correct += 1
    print(f"{text:<45} {expected:<35} {predicted:<35} {ok}")

print(f"\nAccuracy: {correct}/{len(test_cases)}")


Input                                         Expected                            Predicted                           OK
------------------------------------------------------------------------------------------------------------------------
Hi! How are you?                              greeting                            greeting                            ✓
Goodbye, thank you!                           goodbye                             goodbye                             ✓
Thanks so much for your help                  gratitude                           gratitude                           ✓
I feel very anxious and I cannot sleep at night asking_mental_health_question       asking_mental_health_question       ✓
How do I manage panic attacks?                asking_mental_health_question       asking_mental_health_question       ✓
What is the capital of France?                out_of_scope                        out_of_scope                        ✓
أنا أشعر بالاكتئاب الشديد           